# Design Patterns 08: Circuit Breaker

Implementing a state-based failure protection mechanism.

In [ ]:
import time
import random

class CircuitBreaker:
    def __init__(self, failure_threshold=3, recovery_timeout=2):
        self.failure_threshold = failure_threshold
        self.recovery_timeout = recovery_timeout
        self.failures = 0
        self.state = "CLOSED" 
        self.last_failure_time = 0

    def call(self, func, *args):
        if self.state == "OPEN":
            if time.time() - self.last_failure_time > self.recovery_timeout:
                print("Circuit is HALF-OPEN. Testing service...")
                self.state = "HALF-OPEN"
            else:
                print("Circuit is OPEN. Request blocked.")
                return None

        try:
            result = func(*args)
            self._reset()
            return result
        except Exception as e:
            self._record_failure()
            print(f"Call failed: {e}")

    def _record_failure(self):
        self.failures += 1
        self.last_failure_time = time.time()
        if self.failures >= self.failure_threshold:
            self.state = "OPEN"
            print("Failure threshold reached. Opening Circuit!")

    def _reset(self):
        self.failures = 0
        self.state = "CLOSED"
        print("Call success. Circuit CLOSED.")

# Simulating a flaky service
def risky_service():
    if random.choice([True, False]):
        raise ConnectionError("Service Down")
    return "Data Received"

cb = CircuitBreaker(failure_threshold=2, recovery_timeout=1)

for i in range(5):
    print(f"\nAttempt {i+1}:")
    cb.call(risky_service)
    time.sleep(0.5)